In [1]:
from dotenv import load_dotenv

import os
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b",api_key=groq_api_key
)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10f41acf0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10f41b770>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi, My name is Venu")])

AIMessage(content='Hello Venu! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi, My name is Venu". We need to respond appropriately, greeting and perhaps ask how can help. No policy issues. Just a friendly response.'}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 78, 'total_tokens': 139, 'completion_time': 0.127234102, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.003387455, 'prompt_tokens_details': None, 'queue_time': 0.367684568, 'total_time': 0.130621557}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_ca5edfaab2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057ea-789e-7aa0-9497-547948c4a83b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 61, 'total_tokens': 139, 'output_token_details': {'reasoning': 36}})

In [3]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="Hi, My name is Venu"),
        AIMessage(content="Hello Venu! Nice to meet you. How can I assist you today?"),
        HumanMessage(content="What is my name?")
    ]
)

AIMessage(content='Your name is Venu.', additional_kwargs={'reasoning_content': 'The user asks "What is my name?" The assistant should recall that the user introduced themselves as Venu. So answer: Your name is Venu.'}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 109, 'total_tokens': 156, 'completion_time': 0.097891052, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.004988665, 'prompt_tokens_details': None, 'queue_time': 0.307558309, 'total_time': 0.102879717}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_3166198c1d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057ea-7af3-7e92-9d0e-5e68f4a81486-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 109, 'output_tokens': 47, 'total_tokens': 156, 'output_token_details': {'reasoning': 32}})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [4]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model,get_session_history)

/var/folders/h_/3_y8qn2x5t5_7mw2r32c4dp40000gn/T/ipykernel_12808/3287025537.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
/Users/venu/Documents/AI/LangChain/venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


ChatMessageHistory is an in-memory container for messages. It holds a sequence such as:

[

    HumanMessage(content="Hi, My name is Venu"),

    AIMessage(content="Hello, Venu!"),
    
]

RunnableWithMessageHistory is the wrapper that connects the model, the session ID, and the appropriate saved ChatMessageHistory.

In [5]:
"""
store = {
    "chat1": ChatMessageHistory(...),
    "chat2": ChatMessageHistory(...),
}
"""

'\nstore = {\n    "chat1": ChatMessageHistory(...),\n    "chat2": ChatMessageHistory(...),\n}\n'

In [6]:
config1 = {"configurable":{"session_id":"chat1"}}

In [7]:
with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Venu")],
    config=config1
)

AIMessage(content='Hello Venu! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond. The user says "Hi, My name is Venu". We should greet them and ask how can help. Follow policies. No special constraints.'}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 78, 'total_tokens': 138, 'completion_time': 0.126784223, 'completion_tokens_details': {'reasoning_tokens': 35}, 'prompt_time': 0.003779897, 'prompt_tokens_details': None, 'queue_time': 0.366178646, 'total_time': 0.13056412}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_93703442d9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057ea-7cf1-70f2-a6a1-db9b87bc4108-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 60, 'total_tokens': 138, 'output_token_details': {'reasoning': 35}})

In [8]:
with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config1
)

AIMessage(content='Your name is Venu.', additional_kwargs={'reasoning_content': 'The user asks "What is my name?" The conversation: system gave instructions about refusing etc. The user introduced themselves as Venu. So answer: Venu.'}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 109, 'total_tokens': 158, 'completion_time': 0.103513784, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.005055744, 'prompt_tokens_details': None, 'queue_time': 0.604337658, 'total_time': 0.108569528}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_60e4b492db', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057ea-805b-7343-a463-2f49c587ac37-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 109, 'output_tokens': 49, 'total_tokens': 158, 'output_token_details': {'reasoning': 34}})

In [9]:
config2 = {"configurable":{"session_id":"chat2"}}
with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config2
)

AIMessage(content='I’m not able to see your name, so I don’t know it. If you’d like to share it, feel free to let me know!', additional_kwargs={'reasoning_content': 'The user asks "What is my name?" There\'s no prior context. The system says we are ChatGPT. We have no personal data. According to policy, we must not reveal personal data we don\'t have. We can say we don\'t know. So answer: I don\'t know your name. Ask if they\'d like to tell.'}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 76, 'total_tokens': 182, 'completion_time': 0.22114004, 'completion_tokens_details': {'reasoning_tokens': 66}, 'prompt_time': 0.031934038, 'prompt_tokens_details': None, 'queue_time': 0.315696841, 'total_time': 0.253074078}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_3166198c1d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057ea-8365-7440-a896-9816582975dc-0', tool_calls=[], inval

In [10]:
with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Venu")],
    config=config2
)

AIMessage(content='Nice to meet you, Venu! How can I help you today?', additional_kwargs={'reasoning_content': 'We need to respond appropriately. The user says "Hi, My name is Venu". Should respond acknowledging. No policy issue.'}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 124, 'total_tokens': 175, 'completion_time': 0.108229844, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.083975055, 'prompt_tokens_details': None, 'queue_time': 0.333045008, 'total_time': 0.192204899}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_47082602e2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057ea-85dc-7fc0-9baf-ebfbbec5d0cd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 124, 'output_tokens': 51, 'total_tokens': 175, 'output_token_details': {'reasoning': 27}})

In [11]:
with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config2
)

AIMessage(content='Your name is Venu.', additional_kwargs={'reasoning_content': 'The user asks "What is my name?" We have context: they said "Hi, My name is Venu". So answer: Venu. Should comply.'}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 154, 'total_tokens': 203, 'completion_time': 0.101143836, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.007606576, 'prompt_tokens_details': None, 'queue_time': 0.28036951, 'total_time': 0.108750412}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be4c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057ea-8843-7423-8c82-e12f3a18f1b2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 154, 'output_tokens': 49, 'total_tokens': 203, 'output_token_details': {'reasoning': 34}})

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [12]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompts = ChatPromptTemplate.from_messages(
    [
        ("system","You are a AI Assistant, Answer the question for the following"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompts|model

In [13]:
chain.invoke({"messages":[HumanMessage(content="Hi, My name is Venu")]})

AIMessage(content='Hello Venu! Nice to meet you. I’m here to help with any questions or tasks you have. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says: "Hi, My name is Venu". Probably they are greeting. We should respond politely, introduce ourselves, ask how can help. No disallowed content. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 93, 'total_tokens': 172, 'completion_time': 0.16601718, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.004438178, 'prompt_tokens_details': None, 'queue_time': 0.311926626, 'total_time': 0.170455358}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_60e4b492db', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057ea-8a40-70e1-a8d2-0a742e337edd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 79, 'total_tokens': 172,

In [14]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

/Users/venu/Documents/AI/LangChain/venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [15]:
config3 = {"configurable":{"session_id":"chat3"}}
with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Venu")],
    config=config3
)

AIMessage(content='Hello Venu! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi, My name is Venu". Likely a greeting, we should respond politely, maybe ask how can help. Follow guidelines.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 93, 'total_tokens': 150, 'completion_time': 0.122670107, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.004964542, 'prompt_tokens_details': None, 'queue_time': 0.314321468, 'total_time': 0.127634649}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_1d982b31b2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057ea-8cc9-7593-a8ed-93325b169335-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 57, 'total_tokens': 150, 'output_token_details': {'reasoning': 32}})

In [16]:
##Adding more complexity

prompts = ChatPromptTemplate.from_messages(
    [
        ("system","You are a AI Assistant, Answer the question for the following in {language} language"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompts|model

ChatPromptTemplate.from_messages([...]) builds a template that produces a list of messages when filled in, instead of a plain string. Each item in the list can be:

A tuple (role, text) — a static message with a fixed role. ("system", "...") becomes a SystemMessage when rendered.

A MessagesPlaceholder — a "hole" that gets replaced at invocation time by a list of messages you supply, not a fixed string.

MessagesPlaceholder(variable_name="messages") tells the template: "When someone calls .invoke({...}), look for a key called "messages" in the input dict, and splice whatever list of BaseMessage objects is there directly into this position."

So when you call:

chain.invoke({"messages": [HumanMessage(content="Hi, My name is Venu")]})

internally ChatPromptTemplate.invoke does roughly:

resolved_messages = [

    SystemMessage(content="You are a AI Assistant, Answer the question for the following"),

] + input_dict["messages"]

= [SystemMessage(...), HumanMessage(content="Hi, My name is Venu")]

This produces a ChatPromptValue, which wraps that message list and exposes .to_messages() for the next step in the chain.

chain = prompts | model

The | operator is LangChain's LCEL (LangChain Expression Language) compose operator. It builds a RunnableSequence where the output of the left side becomes the input of the right side:

chain.invoke(x) == model.invoke(prompts.invoke(x))

In [17]:
response = chain.invoke({"messages":[HumanMessage(content="Hi, My name is Venu")],"language":"Telugu"})

In [18]:
response.content

'నమస్తే వేను! మీతో మాట్లాడటం చాలా ఆనందంగా ఉంది. నేను మీకు ఎలా సహాయం చేయగలను?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [19]:
with_message_history =RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config4 = {"configurable":{"session_id":"chat4"}}
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="Hi, My name is Venu")],
    "language":"Telugu"},
    config=config4
)

/Users/venu/Documents/AI/LangChain/venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [20]:
response.content

'హాయ్, వేను! మీతో కలుసుకోవడం ఆనందంగా ఉంది. నేను మీకు ఎలా సహాయం చేయగలను?'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.


'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [21]:
from langchain.messages import HumanMessage,trim_messages,SystemMessage

trimmer = trim_messages(
    max_tokens=45,
    token_counter=model,
    strategy="last",
    allow_partial=False,
    include_system=True,
    start_on="human")

messages = [
        SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

/Users/venu/Documents/AI/LangChain/venv/lib/python3.14/site-packages/langchain_core/language_models/base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
/Users/venu/Documents/AI/LangChain/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

#itemgetter("messages") is a small function that, given a dict, extracts the value at key "messages".
#  It's equivalent to writing lambda d: d["messages"], just more efficient/idiomatic.

#RunnablePassthrough is a LangChain Runnable that normally just forwards its input unchanged. 
# Its .assign(...) method lets you add/overwrite specific keys in the input dict while keeping 
# the rest untouched.

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |prompts
    |model
)
#This builds a pipeline that automatically trims the chat history before sending it to the model,
# then runs a full request.

response = chain.invoke({
    "messages": messages + [HumanMessage(content="What ice cream i like?")],
    "language": "English"
    })

In [31]:
"""
Stage A: RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)

Input to the whole chain is a dict: {"messages": [...], "language": "English"}.
RunnablePassthrough.assign(...) keeps all existing keys (messages, language) and 
overwrites messages with a new computed value.
The new value comes from itemgetter("messages") | trimmer:
itemgetter("messages") pulls the raw messages list out of the input dict.
That raw list is piped into trimmer (the trim_messages runnable created earlier),
 which trims it down based on max_tokens=45, strategy="last", include_system=True, etc.
Result of Stage A: {"messages": <trimmed messages>, "language": "English"} — same shape as input,
 but messages is now shorter.
 
"""

'\nStage A: RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)\n\nInput to the whole chain is a dict: {"messages": [...], "language": "English"}.\nRunnablePassthrough.assign(...) keeps all existing keys (messages, language) and \noverwrites messages with a new computed value.\nThe new value comes from itemgetter("messages") | trimmer:\nitemgetter("messages") pulls the raw messages list out of the input dict.\nThat raw list is piped into trimmer (the trim_messages runnable created earlier),\n which trims it down based on max_tokens=45, strategy="last", include_system=True, etc.\nResult of Stage A: {"messages": <trimmed messages>, "language": "English"} — same shape as input,\n but messages is now shorter.\n\n'

In [23]:
response

AIMessage(content='I’m not sure which flavors you enjoy—could you tell me a bit about your favorite tastes?  \nFor example:\n\n* Do you prefer classic flavors like vanilla, chocolate, or strawberry?  \n* Are you into something a little more adventurous, like mint‑chip, coffee, or cookie‑dough?  \n* Do you like fruit‑based scoops (mango, raspberry) or nutty, caramel‑swirled options?  \n\nLet me know what you usually gravitate toward, and I can suggest some ice‑cream picks you might love!', additional_kwargs={'reasoning_content': 'The user asks: "What ice cream i like?" Probably they want the assistant to guess or ask clarification. We don\'t have prior info. We should ask for preferences. We can respond politely, ask about flavors, etc.'}, response_metadata={'token_usage': {'completion_tokens': 168, 'prompt_tokens': 153, 'total_tokens': 321, 'completion_time': 0.354635695, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.006068877, 'prompt_tokens_details': None, '

In [24]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked for the result of\u202f**2\u202f+\u202f2**.'

In [25]:
##let's wrap it in a message history

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

/Users/venu/Documents/AI/LangChain/venv/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [29]:
config5 = {"configurable":{"session_id":"chat5"}}
with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="what is my name?")],
        "language": "English",
    },
    config=config4
)

AIMessage(content='I don’t have any information about your name. If you’d like me to address you by a particular name, just let me know!', additional_kwargs={'reasoning_content': 'The user asks "what is my name?" No prior info given. Should respond that I don\'t know. According to policy, we can ask for clarification or say I don\'t know. So answer politely.'}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 157, 'total_tokens': 235, 'completion_time': 0.167822299, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.006871134, 'prompt_tokens_details': None, 'queue_time': 0.478853855, 'total_time': 0.174693433}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_0708ac49a5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057eb-3b51-7ca0-bc73-546dc9042450-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 78, 'total_t

In [27]:
with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    },
    config=config5
)

AIMessage(content='You asked for the result of\u202f**2\u202f+\u202f2**.', additional_kwargs={'reasoning_content': 'The user asks: "what math problem did i ask". We should answer: they asked "2 + 2". Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 153, 'total_tokens': 205, 'completion_time': 0.111462247, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.023212115, 'prompt_tokens_details': None, 'queue_time': 0.371861628, 'total_time': 0.134674362}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a057ea-a6c7-7501-910b-255e74b5f32f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 52, 'total_tokens': 205, 'output_token_details': {'reasoning': 28}})